# 🧠 CerebroSpheroid V5 Small: Rectified Contiguous TPU Pretraining (100M Params)

This notebook trains **CerebroSpheroid V5 Small** (105.9M parameters) on Google Colab Cloud TPU (v2-8, v3-8, v4-8, v5e, v6e) using a **rectified contiguous pretraining pipeline**.

### 🌟 Technical Highlights & Fixes
1. **100% Loss-Bearing Contiguous Tokens (Zero Wasted Prompt Tokens):**
   - In standard pretraining, every single token in the sequence receives next-token prediction loss.
   - Eliminates the previous prompt/target slice mismatch where 50%+ of tokens were left out of loss.
   - Eliminates artificial prompt prepending (`"Study and comprehend..."`).
   - Packs raw documents contiguously with EOS delimiters into full **4,096-token sequence blocks**.
2. **Audited 70/15/7/5/3 Token-Weighted Mixture:**
   - **70% Broad Clean Web Text:** `HuggingFaceFW/fineweb` (sample-10BT) — general language bedrock.
   - **15% Academic & Educational:** `HuggingFaceFW/fineweb-edu` (score $\ge 3.0$) + `cosmopedia-v2`.
   - **7% Pretraining Code:** Indented clean Python / code repositories.
   - **5% Mathematical Web:** `HuggingFaceTB/finemath` (finemath-4plus) / `OpenWebMath`.
   - **3% Measured Long-Context Retrieval:** Synthetically placed needles and keys in 4096-token documents with held-out validation.
3. **Strict Separation of Pretraining vs. SFT:**
   - Chat dialogues (SmolTalk) and agent tool traces are strictly moved to Post-Training SFT.
   - Base pretraining learns pure grammar, world knowledge, reasoning, and code.
4. **Capacity-Matched Token Budget (1B to 3B Loss-Bearing Tokens):**
   - For 105.9M parameters, 1B–3B tokens is the optimal pretraining regime.
   - At $64 \times 4096 = 262,144$ tokens/step, 1B tokens is reached in ~3,800 steps (~3 hours on TPU v5e/v3).

## 1. Verify Cloud TPU Hardware & JAX Devices


In [ ]:
import jax
devices = jax.devices()
print(f"Detected {len(devices)} device(s): {devices[0].platform.upper()}")
for idx, d in enumerate(devices):
    print(f"  [{idx}] {d}")

if devices[0].platform != "tpu":
    print("\n⚠️ Note: You are running on CPU/GPU. To use a TPU, select: Runtime -> Change runtime type -> TPU")
else:
    print("\n🚀 Connected to Cloud TPU hardware accelerator! Ready for BF16 systolic execution.")

## 2. Clone Repository & Install Dependencies
Clones the official  repository into the Colab environment and installs dependencies.


In [ ]:
import os, sys

REPO_DIR = "/content/cerebro_spheroid_v5"
REPO_URL = "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git"

if not os.path.exists(REPO_DIR):
    print("🔄 Cloning CerebroSpheroid V5 repository...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("🔄 Repository exists. Pulling latest commits...")
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install prerequisites
!pip install -q transformers datasets pyarrow optax torch
print(f"✓ Environment ready. Working directory: {os.getcwd()}")


## 3. Connect to Cloud Sync Hub (bioflo.in)
Verifies connectivity to your website sync hub () for automatic two-way checkpoint synchronization and live token/loss tracking (no Google Drive required).


In [ ]:
from src.cloud_sync import check_remote_manifest, DEFAULT_URL

print(f"Checking Cloud Sync Hub at: {DEFAULT_URL}...")
status = check_remote_manifest('v5_small')
print(f"✓ Hub Status: {status.get('status', 'unknown').upper()}")
print(f"  - Has Remote Latest: {status.get('has_latest', False)}")
print(f"  - Has Remote Best:   {status.get('has_best', False)}")
print(f"  - Active Models:     {list((status.get('manifest') or {}).keys())}")


## 4. Run Architecture Sanity Check (105.89M Parameters)

In [ ]:
import jax
import jax.numpy as jnp
from src.jax.v5_model import ModelConfigV5, PRESETS_V5, init_params_v5

cfg_dict = PRESETS_V5['v5_small'].copy()
cfg_dict['sensory_position_steps'] = 4096
cfg_dict['max_phase_steps'] = 4096
cfg_dict['max_seq_len'] = 4096
cfg_dict['use_rope'] = True

config = ModelConfigV5(**cfg_dict)
key = jax.random.PRNGKey(42)
params, buffers = init_params_v5(key, config)

total_params = sum(x.size for x in params.values())
print(f"✓ CerebroSpheroid V5 Small initialized successfully!")
print(f"  - Shells:                {config.num_shells}")
print(f"  - Surface Dim:           {config.surface_dim}")
print(f"  - Radial Dim:            {config.radial_dim}")
print(f"  - Shortcut Tracts:       {config.num_shortcut_tracts}")
print(f"  - Sensory Pos Horizon:   {config.sensory_position_steps} tokens")
print(f"  - Motor Phase Horizon:   {config.max_phase_steps} tokens")
print(f"  - Total Parameters:      {total_params / 1e6:.2f}M")

## 5. Launch Full-Dataset Pretraining (60,000 Steps = 15.7 Billion Tokens)

This command launches the high-throughput TPU JAX trainer across the **full dataset**:
- **10x Ingestion Scale:** 60,000 steps ingesting ~15.7 Billion loss-bearing tokens (full FineWeb sample-10BT + Code).
- **100% Loss-Bearing Tokens:** Every single token in the 4096-token sequence is evaluated under cross-entropy.
- **Curated Stage 1 Pretraining:** 85% Broad Web (FineWeb 10BT) + 15% Pretraining Code (The Stack v2).
- **Automatic Cloud Hub Synchronization:** Checkpoints (`latest.pt`, `best.pt`) and live metrics stream directly to your website (`bioflo.in`). If Colab disconnects, simply re-run with `--resume` to continue seamlessly.

In [ ]:
!python scripts/train_v5_small_colab_tpu.py \
    --mode contiguous_pretrain \
    --stage pretrain \
    --total-window 4096 \
    --sensory-position-steps 4096 \
    --max-phase-steps 4096 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 2e-3 \
    --dtype bf16 \
    --total-steps 60000 \
    --save-every 250 \
    --milestone-every 1000 \
    --probe-every 500 \
    --log-every 10 \
    --cloud-sync \
    --resume \
    --checkpoint-dir /content/checkpoints/v5_small_scratch

## 6. (Optional) Stage 2 — Mid-training (Targeted Skills, FineMath & 4k Retrieval)

Once base pretraining is complete, mid-training sharpens STEM reasoning and long-range retrieval:
- **Data Mixture:** 45% FineWeb-Edu (>= 3.5), 25% FineMath, 15% Code, 15% Multi-Position 4k Retrieval Needles.
- **Objective:** Continuous 4096-token next-token prediction starting from the Stage 1 checkpoint.

In [ ]:
!python scripts/train_v5_small_colab_tpu.py \
    --mode contiguous_pretrain \
    --stage midtrain \
    --total-window 4096 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 5e-4 \
    --dtype bf16 \
    --total-steps 15000 \
    --save-every 250 \
    --milestone-every 1000 \
    --probe-every 500 \
    --log-every 10 \
    --cloud-sync \
    --checkpoint-dir /content/checkpoints/v5_small_midtrain

## 7. (Optional) Stage 3 — Supervised Fine-Tuning (SFT) & Tool Calling

Teaches `<think>` deliberation and strict `<act> calc(...) </act>` tool-calling:
- **Loss Masking:** Prompt-masked (loss applies strictly to thinking, answers, and tool calls).
- **Data:** Curated SmolTalk, OpenHermes, and verified calculation traces.

In [ ]:
!python scripts/train_v5_small_colab_tpu.py \
    --mode sft \
    --total-window 4096 \
    --batch-size 8 \
    --grad-accum 2 \
    --lr 2e-4 \
    --dtype bf16 \
    --total-steps 2000 \
    --save-every 200 \
    --probe-every 250 \
    --log-every 10 \
    --cloud-sync \
    --checkpoint-dir /content/checkpoints/v5_small_sft

## 8. Interactive Live Inference & Prompt Testing
Load your trained checkpoint (locally or auto-pulled from bioflo.in) and generate answers to arbitrary prompts.

In [ ]:
import os
import jax
import jax.numpy as jnp
import numpy as np
from src.jax.v5_model import ModelConfigV5, PRESETS_V5, training_rollout
from src.jax.checkpoint import load_torch_pt_to_jax
from src.cloud_sync import pull_checkpoint
from src.corpus import build_tokenizer, format_stimulus_prompt, clean_motor_response, encode_no_special

CKPT_PATH = "/content/checkpoints/v5_small_scratch/v5_small_scratch_best.pt"
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = "/content/checkpoints/v5_small_scratch/v5_small_scratch_latest.pt"
if not os.path.exists(CKPT_PATH):
    print("🌐 Checkpoint not found locally. Pulling best.pt from bioflo.in...")
    pull_checkpoint(CKPT_PATH, target_type="best", model="v5_small")

print(f"Loading checkpoint: {CKPT_PATH}...")
tokenizer = build_tokenizer("llama")

cfg_dict = PRESETS_V5["v5_small"].copy()
cfg_dict["sensory_position_steps"] = 4096
cfg_dict["max_phase_steps"] = 4096
cfg_dict["max_seq_len"] = 4096
config = ModelConfigV5(**cfg_dict)

params, buffers, meta = load_torch_pt_to_jax(CKPT_PATH, config, target_dtype=jnp.bfloat16)
print(f"✓ Loaded model from Step {meta.get('step', 0):,} (Loss: {meta.get('loss', 0.0):.4f})")

def generate(prompt_text: str, max_tokens: int = 128) -> str:
    stimulus = format_stimulus_prompt(prompt_text)
    tokens = encode_no_special(tokenizer, stimulus)
    prompt_arr = jnp.array([tokens], dtype=jnp.int32)
    
    out_logits = training_rollout(
        params=params,
        prompts=prompt_arr,
        target_len=max_tokens,
        buffers=buffers,
        config=config,
        targets=None,
        feedback_tokens=None,
        pure_teacher_forcing=False,
        deliberation_mode="teacher_forcing",
        repetition_penalty=1.15,
    )
    gen_token_ids = np.array(jnp.argmax(out_logits, axis=-1)[0])
    raw = tokenizer.decode(gen_token_ids, skip_special_tokens=False)
    return clean_motor_response(raw).strip()

# Test Prompts
test_prompt = "What is the capital of France?"
print(f"\nPrompt:   {test_prompt}")
print(f"Response: {generate(test_prompt)}")